In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('dataset.csv')


In [10]:
df.columns

Index(['Test Sample', 'Experimental Run', 'Infill Density (%)',
       'Nozzle Temperature (C)', 'Infill Pattern', 'Mean Tensile Stress (MPa)',
       'Tensile Stress SD (MPa)'],
      dtype='object')

In [4]:
import pandas as pd
import numpy as np

def augment_data_vectorized(df, n_synthetic=10):
    """
    Expands the dataset by generating synthetic samples using Gaussian noise.
    
    Args:
        df: The original DataFrame.
        n_synthetic: Number of synthetic points to generate per original row.
    """
    # 1. Repeat each row n_synthetic times
    df_aug = df.loc[df.index.repeat(n_synthetic)].reset_index(drop=True)
    
    # 2. Generate Gaussian noise: Mean = 0, SD = 1
    # We multiply this by the actual SD from the table to scale the noise
    noise = np.random.normal(size=len(df_aug))
    
    # 3. Apply the noise: New Value = Mean + (Noise * SD)
    df_aug['Tensile Stress (Augmented)'] = (
        df_aug['Mean Tensile Stress (MPa)'] + 
        (noise * df_aug['Tensile Stress SD (MPa)'])
    ).round(3)
    
    # 4. Shuffle the dataset so the model doesn't learn the order of experiments
    return df_aug.sample(frac=1).reset_index(drop=True)

# Execute the augmentation
# If you have 33 rows, this will result in 330 rows
augmented_df = augment_data_vectorized(df, n_synthetic=10)

# Display a preview
print(f"Dataset expanded from {len(df)} to {len(augmented_df)} rows.")
print(augmented_df[['Infill Pattern', 'Mean Tensile Stress (MPa)', 'Tensile Stress (Augmented)']].head())

Dataset expanded from 33 to 330 rows.
  Infill Pattern  Mean Tensile Stress (MPa)  Tensile Stress (Augmented)
0       Triangle                      12.68                      12.367
1           Grid                      17.44                      16.834
2  Triangle-Hexa                      12.64                      12.495
3           Grid                       8.42                       8.511
4           Grid                      17.76                      16.953


In [12]:
augmented_df.columns

Index(['Test Sample', 'Experimental Run', 'Infill Density (%)',
       'Nozzle Temperature (C)', 'Infill Pattern', 'Mean Tensile Stress (MPa)',
       'Tensile Stress SD (MPa)', 'Tensile Stress (Augmented)'],
      dtype='object')

In [17]:
augmented_df

,Test Sample,Experimental Run,Infill Density (%),Nozzle Temperature (C),Infill Pattern,Mean Tensile Stress (MPa),Tensile Stress SD (MPa),Tensile Stress (Augmented)
0,21,3,60.00,240.00,Triangle,12.68,0.42,12.367
1,6,24,80.00,240.00,Grid,17.44,0.82,16.834
2,31,10,60.00,240.00,Triangle-Hexa,12.64,0.43,12.495
3,1,14,45.86,225.86,Grid,8.42,0.31,8.511
4,8,32,60.00,260.00,Grid,17.76,0.75,16.953
...,...,...,...,...,...,...,...,...
325,31,10,60.00,240.00,Triangle-Hexa,12.64,0.43,12.865
326,31,10,60.00,240.00,Triangle-Hexa,12.64,0.43,13.102
327,29,33,60.00,220.00,Triangle-Hexa,12.13,0.40,11.969
328,20,25,60.00,240.00,Triangle,11.74,0.34,11.290


In [13]:
# 1. Apply One-Hot Encoding to the Infill Pattern column
df_encoded = pd.get_dummies(augmented_df, columns=['Infill Pattern'], prefix='Pattern', dtype=int)

# 2. View the new columns
print("New Columns:", [col for col in df_encoded.columns if 'Pattern' in col])

# 3. Preview the transformed data
display(df_encoded.head())

New Columns: ['Pattern_Grid', 'Pattern_Triangle', 'Pattern_Triangle-Hexa']


,Test Sample,Experimental Run,Infill Density (%),Nozzle Temperature (C),Mean Tensile Stress (MPa),Tensile Stress SD (MPa),Tensile Stress (Augmented),Pattern_Grid,Pattern_Triangle,Pattern_Triangle-Hexa
0,21,3,60.00,240.00,12.68,0.42,12.367,0,1,0
1,6,24,80.00,240.00,17.44,0.82,16.834,1,0,0
2,31,10,60.00,240.00,12.64,0.43,12.495,0,0,1
3,1,14,45.86,225.86,8.42,0.31,8.511,1,0,0
4,8,32,60.00,260.00,17.76,0.75,16.953,1,0,0
